In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

In [ ]:
df = pd.read_csv("mobile_usage_dataset.csv")
df.head()


In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
(df.isnull().sum()/len(df))*100

In [ ]:
df['Device_Brand'] =df['Device_Brand'].fillna(df['Device_Brand'].mode()[0])

In [ ]:
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')

In [ ]:
df['Age'] =df['Age'].fillna(df['Age'].median())
df['Age'].round().astype('Int64')



In [ ]:
df['Social_Media_Hours'] =df['Social_Media_Hours'].fillna(df['Social_Media_Hours'].median())


In [ ]:
df['Work_Hours'] =df['Work_Hours'].fillna(df['Work_Hours'].median())


In [ ]:
df.info()
df.isnull().sum()

In [ ]:
df.info()

In [ ]:
num_cols = ['Age','Social_Media_Hours','Gaming_Hours','Work_Hours','Streaming_Hours','Total_Screen_Time','High_Usage']

plt.figure(figsize=(8,8))

for col in num_cols:
    sns.boxplot(x=df[col])
    plt.title(f'Box plot of {col}')
    plt.show()

In [ ]:
df[num_cols].describe() #to check max-min values and logically impossible ones.

In [ ]:
df['Social_Media_Hours'] = np.where(df['Social_Media_Hours']>24,24,df['Social_Media_Hours']) #capping the max value

In [ ]:
df['Social_Media_Hours'].max() #to check the max value

In [ ]:
#checks co-linearity b/w numeric values (data), hence checking linear dependence

plt.figure(figsize=(8,5))
sns.heatmap(df.corr(numeric_only=True),annot=True,cmap='coolwarm',fmt='.2f')
plt.title("Correlation matrix")
plt.show()

In [ ]:
df.info()

In [ ]:
#removing the duplicate casing issues

df['Gender'] = df['Gender'].str.lower()
df['Device_Brand'] = df['Device_Brand'].str.lower()
df['Location_Type'] = df['Location_Type'].str.lower()


In [ ]:
#regression pipeline

df_encoded = pd.get_dummies(df,drop_first=True) #getting copy of the data frame into a variable for computation.
df_encoded.head(10)

In [ ]:
df.info()
df_encoded.shape

In [ ]:
#declaring x and y variables 
y = df_encoded['Total_Screen_Time'] #target variable

x = df_encoded.drop(['Total_Screen_Time','High_Usage'],axis=1)

print("x shape: ", x.shape)
print("y shape: ", y.shape)


In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=42)

print('x_train shape: ',x_train.shape)
print('x_test shape: ',x_test.shape)
print('y_train shape: ',y_train.shape)
print('y_test shape: ',y_test.shape)


In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.fit_transform(x_test)

type(x_train_scaled)
type(x_test_scaled)





In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()

model.fit(x_train_scaled,y_train)
print("model trained")

In [ ]:
y_pred = model.predict(x_test_scaled)
print(y_pred[:5])

In [ ]:
print("Actual values: ",y_test[:5].values)

print("Predicted values: ",y_pred[:5])


In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

mae = mean_absolute_error(y_test,y_pred)
mse = np.sqrt(mean_squared_error(y_test,y_pred))
rscore = r2_score(y_test,y_pred)

print("MAE: ",mae)
print("MSE: ",mse)
print("RSCORE: ",rscore)

#0.80-r2score(very strong-nearly 80% accuracy)

In [ ]:
df_encoded.info

In [ ]:
#classification (logistic regression) 

y_clf = df_encoded['High_Usage']

x_clf = df_encoded.drop(['Total_Screen_Time','High_Usage'],axis=1) #axis=0:rows ; axis=1:columns

print('x1 shape: ',x_clf.shape)
print('y1 shape: ',y_clf.shape)


In [ ]:
x_clf_train,x_clf_test,y_clf_train,y_clf_test = train_test_split(x_clf,y_clf,test_size=0.2,random_state=42)

print('x_clf_train',x_clf_train.shape)
print('x_clf_test',x_clf_test.shape)
print('y_clf_train',y_clf_train.shape)
print('y_clf_test',y_clf_test.shape)

In [ ]:
print(y_clf.value_counts()) #checks the balance of features (0 and 1 a,ong the data)

In [ ]:
sclaer_clf = StandardScaler()

x_clf_train_scaled = sclaer_clf.fit_transform(x_clf_train)
x_clf_test_scaled = sclaer_clf.fit_transform(x_clf_test)

In [ ]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(class_weight='balanced',max_iter=1000) #class weight balanced gives more weight to features that are significantly lesser in num as compared to the other one.
log_model.fit(x_clf_train_scaled,y_clf_train)

print("training done!")

In [ ]:
y_pred_clf = log_model.predict(x_clf_test_scaled)

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

accu = accuracy_score(y_clf_test,y_pred_clf)
print('Accuracy: ',accu)

cm = confusion_matrix(y_clf_test,y_pred_clf)
print("confusion_matrix:\n ",cm)

print(classification_report(y_clf_test,y_pred_clf))

In [51]:
#Random forest for linear resgression
from sklearn.ensemble import RandomForestRegressor

rf_reg = RandomForestRegressor(n_estimators=100,random_state=42)
rf_reg.fit(x_train,y_train)
y_pred_rf = rf_reg.predict(x_test)